# MAKRO 2026 in ModelFlow: the effect of a shock on the input-output system

This notebook shows how one of DREAM's standard shocks changes the input-output (IO) system of MAKRO in a chosen
year: which deliveries rise and fall, from domestic production and from imports, how the import content of
final demand changes, and how the change in the sector prices passes through to the prices of the demand
components.

It reads `data\makro26_shocks.pcimz`, written by `makro26_all_shocks.ipynb` (section 8): the model, the
baseline and every solved shock as kept solutions. Nothing is solved. The charts come from
`tools-io\io_viz.py`; `makro26_io_viz.ipynb` shows the IO system of the baseline itself.

It needs ModelFlow from `C:\modelflow2\modelflow` and `plotly` (both in `book314b`).

In [1]:
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from modelclass import model
sys.path.insert(0, r'C:\mfmodeller_raw\makro\makro26\tools-io')
sys.path.insert(0, r'C:\mfmodeller_raw\makro\makro26\tools')
import io_viz, dream_shocks
pd.set_option('display.max_colwidth', 120, 'display.max_rows', 200)

## 1. The model and the kept solutions

The file is large (260 MB) and loads in about 10 seconds. `keep_solutions` holds the baseline and one dataframe
per shock.

In [2]:
mshocks, _ = model.modelload(r'C:\mfmodeller_raw\makro\makro26\data\makro26_shocks.pcimz')
name_of = {label: name for name, label in dream_shocks.SHORT.items()}
pd.DataFrame({'What DREAM changes': {label: dream_shocks.DESCRIPTION.get(name_of.get(label), '')
                                     for label in mshocks.keep_solutions if label != 'Baseline'}})

Feather/zip file read: C:\mfmodeller_raw\makro\makro26\data\makro26_shocks.pcimz


Model: MAKRO 2026-September (DREAM) in ModelFlow, baseline and DREAM's standard shocks 2030-2129


,What DREAM changes
Labour productivity,"Labour productivity (qProdHh_t, qProdxDK) up 1 per cent"
Labour supply,Structural employment (snLHh[a]) up 1 per cent; the participation parameter uDeltag is freed
Export market,"Export market (uXMarked) up, 1 per cent of GDP"
Import prices,Import prices (pM[s]) up 1 per cent
Capital productivity,"Capital productivity (uK[iM,sp], uK[iB,sp]) up 1 per cent"
Public employment,Public employment (hL[off]) up by 1 per cent of GDP
Public purchases,Public purchases of goods and services (qR[off]) up by 1 per cent of GDP
Public investment,"Public investments (qI_s[i,off]) up by 1 per cent of GDP"
Transfers to households,Lump sum transfer to households (vOffTilHhRest) of 1 per cent of GDP
Interest rate,ECB interest rate (rRenteECB) up 1 percentage point


## 2. Settings

`experiment` is one of the names above; the public expenditure shocks are `'Public purchases'`,
`'Public employment'` and `'Public investment'`. All the shocks are permanent from 2030. `year` is the year
shown.

In [3]:
experiment = 'Public purchases'
year = 2040
min_flow = 0.1        # changes smaller than this (billion kr.) are left out of the Sankey

In [4]:
base_df = mshocks.keep_solutions['Baseline']
alt_df = mshocks.keep_solutions[experiment]
base, alt = io_viz.extract_makro(base_df), io_viz.extract_makro(alt_df)              # values, billion kr.
base_q, alt_q = io_viz.extract_makro(base_df, 'Q'), io_viz.extract_makro(alt_df, 'Q')  # volumes

print(f'{experiment}: {dream_shocks.DESCRIPTION.get(name_of.get(experiment), "")}')
headline = {'QBNP': 'GDP, volume', 'VBNP': 'GDP, value', 'QR__OFF': 'Public purchases (intermediate use, public sector), volume',
            'QM__TOT': 'Imports, volume', 'VM__TOT': 'Imports, value', 'PY__TOT': 'Output price, all sectors'}
h = pd.DataFrame({'': headline, 'baseline': base_df.loc[year, list(headline)], 'experiment': alt_df.loc[year, list(headline)]})
h['difference'] = h['experiment'] - h['baseline']
h['per cent'] = 100 * h['difference'] / h['baseline']
h.style.format({'baseline': '{:,.2f}', 'experiment': '{:,.2f}', 'difference': '{:,.2f}', 'per cent': '{:.3f}'}).set_caption(f'{experiment}, {year}')

Public purchases: Public purchases of goods and services (qR[off]) up by 1 per cent of GDP


,,baseline,experiment,difference,per cent
QBNP,"GDP, volume","2,754.42","2,757.33",2.91,0.106
VBNP,"GDP, value","3,178.61","3,193.22",14.61,0.460
QR__OFF,"Public purchases (intermediate use, public sector), volume",258.90,287.42,28.52,11.017
QM__TOT,"Imports, volume","1,950.82","1,960.26",9.44,0.484
VM__TOT,"Imports, value","2,087.89","2,098.09",10.19,0.488
PY__TOT,"Output price, all sectors",1.15,1.15,0.00,0.260


## 3. The demand components

The change in the deliveries to each demand component in `year`, from domestic production (`dom`) and from
imports (`imp`), in billion kr. at current prices, and in per cent of the baseline. Intermediate use `rXxx` is
the use of the sector `Xxx`.

In [5]:
def by_d(t):
    return t[t['year'] == year].pivot_table(index='d', columns='origin', values='value', aggfunc='sum').fillna(0)


b, a = by_d(base), by_d(alt)
t = (a - b).rename(columns={'dom': 'change, dom', 'imp': 'change, imp'})
t['change, total'] = t.sum(axis=1)
t['per cent'] = 100 * t['change, total'] / b.sum(axis=1).where(b.sum(axis=1) != 0)
t.insert(0, 'baseline', b.sum(axis=1))
t.insert(0, 'group', [io_viz.default_group(d) for d in t.index])
t.insert(0, 'label', [io_viz.DEMAND_LABELS.get(d, '') for d in t.index])
t = t.sort_values(['group', 'baseline'], ascending=[True, False])
lim = t['per cent'].abs().max()
t.style.format({c: '{:,.2f}' for c in t.columns if c.startswith(('baseline', 'change'))} | {'per cent': '{:.2f}'}, na_rep='') \
    .background_gradient(cmap='RdBu', subset=['per cent'], vmin=-lim, vmax=lim) \
    .set_caption(f'{experiment}, {year}: deliveries to the demand components, billion kr.')

origin,label,group,baseline,"change, dom","change, imp","change, total",per cent
d,,,,,,,
cTje,Consumption of services,Consumption,571.80,1.59,0.12,1.71,0.30
cVar,Consumption of goods,Consumption,482.86,1.09,0.23,1.33,0.27
cBol,Consumption of housing,Consumption,312.06,0.97,0.00,0.98,0.31
cEne,Consumption of energy,Consumption,103.53,0.28,0.02,0.29,0.28
cTur,Residents' tourism abroad,Consumption,66.39,0.00,0.29,0.29,0.44
cBil,Consumption of cars,Consumption,63.15,0.13,0.11,0.24,0.38
xVar,Exports of goods,Exports,"1,301.95",-7.90,-0.08,-7.98,-0.61
xTje,Exports of services,Exports,407.03,-4.25,-0.01,-4.26,-1.05
xSoe,Exports of shipping,Exports,360.48,-0.09,-0.00,-0.09,-0.02


## 4. The change in the flows

A Sankey of the changes: from the sectors and imports to the demand components, green where the delivery
rises, red where it falls. The width is the size of the change.

In [6]:
fig = io_viz.shock_sankey(base, alt, year, min_flow=min_flow)
fig.update_layout(title=f'{experiment}: change in the IO flows {year}, billion kr. (green up, red down)')

## 5. The IO matrix

Each cell is the delivery from a sector (column) to a demand component (row), domestic production plus
imports. First the change in billion kr., then in per cent, in values and in volumes. The difference between
the two per cent charts is the change in the prices.

In [7]:
io_viz.heatmap(alt, year, metric='total', base=base, relative=False,
               title=f'{experiment}: change in the IO matrix {year}, billion kr.')

In [8]:
io_viz.heatmap(alt, year, metric='total', base=base, relative=True,
               title=f'{experiment}: change in the IO matrix {year}, values, per cent')

In [9]:
io_viz.heatmap(alt_q, year, metric='total', base=base_q, relative=True,
               title=f'{experiment}: change in the IO matrix {year}, volumes, per cent')

The import share of each cell, the experiment minus the baseline (0.01 = 1 percentage point):

In [10]:
io_viz.heatmap(alt, year, metric='import_share', base=base,
               title=f'{experiment}: change in the import share of each cell {year}')

## 6. Import content of final demand

Direct imports plus the imports used in the domestic production delivered to the component, through all rounds
of intermediate use (see `makro26_io_viz.ipynb`), in the baseline and in the experiment.

In [11]:
ic_b, ic_a = io_viz.import_content(base, year), io_viz.import_content(alt, year)
ic = pd.DataFrame({'label': [io_viz.DEMAND_LABELS.get(d, '') for d in ic_b.index],
                   'baseline, %': 100 * ic_b['import_content'],
                   'experiment, %': 100 * ic_a['import_content'].reindex(ic_b.index)}, index=ic_b.index)
ic['change, pp'] = ic['experiment, %'] - ic['baseline, %']
ic = ic[ic_b['total'] > 0]
lim = ic['change, pp'].abs().max()
display(ic.style.format({c: '{:.2f}' for c in ic.columns if c != 'label'} | {'change, pp': '{:.3f}'})
          .background_gradient(cmap='RdBu', subset=['change, pp'], vmin=-lim, vmax=lim)
          .set_caption(f'{experiment}: import content {year}'))

change = (ic_a - ic_b).reindex(ic.index)
fig = io_viz.import_content_bar(change, title=f'{experiment}: change in the import content {year}')
fig.update_layout(barmode='relative', yaxis_title='percentage points')

,label,"baseline, %","experiment, %","change, pp"
d,,,,
cTur,Residents' tourism abroad,100.00,100.00,0.000
xSoe,Exports of shipping,80.79,80.82,0.033
cBil,Consumption of cars,73.64,73.58,-0.061
xEne,Exports of energy,59.62,59.71,0.086
iL,Change in inventories,59.52,59.47,-0.048
iM,Investment in machinery,55.26,55.24,-0.023
xVar,Exports of goods,52.16,52.23,0.074
cVar,Consumption of goods,49.50,49.45,-0.044
cEne,Consumption of energy,39.31,39.26,-0.056


## 7. Price pass-through

The change in the price of a demand component, to a first order, is the change in the sector prices weighted
by the baseline shares of the sectors in its deliveries:

$$\Delta \log p_d \approx \sum_s w^{dom}_{d,s}\, \Delta \log pY_s + w^{imp}_{d,s}\, \Delta \log pM_s$$

with $\Delta$ the difference between the experiment and the baseline in `year`. The bars show the contribution
of each sector, the diamond the sum. Sectors without imports have no import price.

In [12]:
secs = sorted(io_viz.SECTOR_LABELS)
dlog = lambda v: np.log(alt_df.loc[year, v] / base_df.loc[year, v])
dlog_py = pd.Series({s: dlog(f'PY__{s.upper()}') for s in secs})
dlog_pm = pd.Series({s: dlog(f'PM__{s.upper()}') for s in secs if f'PM__{s.upper()}' in base_df}).dropna()
display(pd.DataFrame({'sector': [io_viz.SECTOR_LABELS[s] for s in secs],
                      'output price pY, %': 100 * dlog_py, 'import price pM, %': 100 * dlog_pm.reindex(secs)})
          .style.format({'output price pY, %': '{:.3f}', 'import price pM, %': '{:.3f}'}, na_rep='')
          .set_caption(f'{experiment}: change in the sector prices {year} (log difference x 100)'))

contrib = io_viz.price_contributions(base, year, dlog_py, dlog_pm)
io_viz.price_passthrough_bar(contrib, title=f'{experiment}: price pass-through to the demand components {year}, per cent')

,sector,"output price pY, %","import price pM, %"
bol,Housing,0.238,
byg,Construction,0.273,
ene,Energy supply,0.178,0.000
fre,Manufacturing,0.184,0.000
lan,Agriculture,0.205,
off,Public,0.117,
soe,Shipping,-0.017,0.000
tje,Services,0.314,0.000
udv,Extraction,0.077,0.000


How good is the first order? The same weights applied to the change in the prices of the IO cells
themselves (`pIOy`, `pIOm`), against the first order from the sector prices:

In [13]:
pb, pa = io_viz.extract_makro(base_df, 'P'), io_viz.extract_makro(alt_df, 'P')
k = ['year', 'd', 's', 'origin']
cell = pb.merge(pa, on=k, suffixes=('_b', '_a'))
cell = cell[cell['year'] == year]
cell['dlogp'] = np.log(cell['value_a'] / cell['value_b'])
w = base[base['year'] == year].merge(cell[k + ['dlogp']], on=k, how='left')
w['w'] = w['value'] / w.groupby('d')['value'].transform('sum')
check = pd.DataFrame({'label': [io_viz.DEMAND_LABELS.get(d, '') for d in sorted(w['d'].unique())]}, index=sorted(w['d'].unique()))
check['first order from pY, pM, %'] = 100 * contrib.groupby('d')['contribution'].sum()
check['from the IO cell prices, %'] = 100 * (w['w'] * w['dlogp'].fillna(0)).groupby(w['d']).sum()
check = check[base[base['year'] == year].groupby('d')['value'].sum().reindex(check.index) > 0]
check.style.format({c: '{:.3f}' for c in check.columns if c != 'label'}).set_caption(f'{experiment}: change in the demand prices {year}')

,label,"first order from pY, pM, %","from the IO cell prices, %"
cBil,Consumption of cars,0.110,0.110
cBol,Consumption of housing,0.240,0.240
cEne,Consumption of energy,0.175,0.199
cTje,Consumption of services,0.276,0.276
cTur,Residents' tourism abroad,0.000,0.000
cVar,Consumption of goods,0.204,0.206
g,Public consumption,0.122,0.122
iB,Investment in buildings,0.269,0.278
iL,Change in inventories,0.131,0.132
iM,Investment in machinery,0.158,0.173


## Another experiment or year

Change `experiment` or `year` in section 2 and run the notebook from there.